# Trabajo Práctico Final — Inteligencia Artificial 2026

**Universidad Nacional Guillermo Brown (UNaB)**  
**Profesor:** Lic. Pablo Moreira  
**Dataset:** KMNIST

---

## Objetivo del Notebook

Este trabajo práctico acompaña el cursado completo de la asignatura. Cada grupo recibe un dataset distinto y debe construir, paso a paso, un sistema de MachineLearning que evolucione desde un modelo baseline hasta una arquitectura compleja, aplicando los conceptos teóricos y prácticos de cada unidad.
El objetivo no es solo que el modelo funcione, sino que elestudiantepueda justificar cada decisión de diseño, interpretar los resultados y aplicar las técnicas vistas en clase para mejorar el sistema.



# Etapa 1 — Exploración y modelo baseline (Obligatorio)

Esta primera etapa debe completarse durante las primeras dos semanas. El objetivo es entender el dataset y establecer un punto de partida medible.

### Carga y análisis exploratorio del dataset (EDA)

- Cargar el dataset usando la fuente indicada en la tabla de asignación.
- Describir las variables: tipos, rangos, distribuciones, valores nulos.
- Visualizar al menos 3 gráficos relevantes (histogramas, distribución de clases, ejemplos de imágenes si corresponde).
- Identificar posibles problemas: desbalance de clases, outliers, data mismatch potencial.

---

### 1.1 Importación de librerías


In [ ]:
%pip install seaborn
%pip install scikit-learn
%pip install requests
%pip install extra-keras-datasets
%pip install tqdm
%pip install datasets

In [10]:
%pip install openml

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached argon2_cffi_bindings-26.1.0-cp310-abi3-win_amd64.whl.metadata (7.5 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached argon2_cffi-25.1.0-py3-none-any.whl (14 kB)
Using cached argon2_cffi_bindings-26.1.0-cp310-abi3-win_amd64.whl (25 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 49.6 MB/s  0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11783 sha256=31e215681c6fa438d66d13cddfe999293a

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
import os
import gzip
import tensorflow as tf
from sklearn.model_selection import train_test_split
import keras
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn
import urllib.request
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Versiones de las librerías instaladas:")
print("- TensorFlow:", tf.__version__)
print("- Keras:", keras.__version__)
print("- Pandas:", pd.__version__)
print("- NumPy:", np.__version__)
print("- Matplotlib:", matplotlib.__version__)
print("- Seaborn:", seaborn.__version__)
print("- urllib.request disponible:", hasattr(urllib.request, "urlopen"))


c:\Users\David\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Versiones de las librerías instaladas:
- TensorFlow: 2.21.0
- Keras: 3.12.4
- Pandas: 2.3.3
- NumPy: 2.2.6
- Matplotlib: 3.10.9
- Seaborn: 0.13.2
- urllib.request disponible: True


In [12]:
import gzip
import os
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Funciones para leer los paquetes binarios comprimidos (.gz)
def cargar_idx_imagenes(ruta_gz):
    with gzip.open(ruta_gz, 'rb') as f:
        f.read(16)  # Saltea el encabezado de control (16 bytes)
        buffer = f.read()
        datos = np.frombuffer(buffer, dtype=np.uint8)
        return datos.reshape(-1, 28, 28)

def cargar_idx_etiquetas(ruta_gz):
    with gzip.open(ruta_gz, 'rb') as f:
        f.read(8)   # Saltea el encabezado de control (8 bytes)
        buffer = f.read()
        return np.frombuffer(buffer, dtype=np.uint8)

# 2. Rutas relativas apuntando al depósito 'data'
# (Usamos os.path.join para que funcione tanto si el notebook está en la raíz como en notebooks/)
ruta_base = 'data' if os.path.exists('data') else '../data'

X_train_raw = cargar_idx_imagenes(os.path.join(ruta_base, 'train-images-idx3-ubyte.gz'))
y_train_raw = cargar_idx_etiquetas(os.path.join(ruta_base, 'train-labels-idx1-ubyte.gz'))
X_test_raw  = cargar_idx_imagenes(os.path.join(ruta_base, 't10k-images-idx3-ubyte.gz'))
y_test_raw  = cargar_idx_etiquetas(os.path.join(ruta_base, 't10k-labels-idx1-ubyte.gz'))

# 3. Concatenar para tener el lote total (70.000 muestras)
X_total = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_total = np.concatenate([y_train_raw, y_test_raw], axis=0)

# 4. Partición estratificada (60% Train, 20% Val, 20% Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_total, y_total, test_size=0.40, random_state=42, stratify=y_total
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 5. Acondicionar para TensorFlow: escala [0.0, 1.0] y canal de profundidad (28, 28, 1)
X_train = (X_train / 255.0).astype(np.float32)[..., np.newaxis]
X_val   = (X_val   / 255.0).astype(np.float32)[..., np.newaxis]
X_test  = (X_test  / 255.0).astype(np.float32)[..., np.newaxis]

print("=== MATERIA PRIMA LISTA PARA TENSORFLOW ===")
print(f"X_train (60%): {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val   (20%): {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_test  (20%): {X_test.shape}  | y_test:  {y_test.shape}")
print(f"Rango de píxeles: [{X_train.min()}, {X_train.max()}] | Tipo de dato: {X_train.dtype}")

=== MATERIA PRIMA LISTA PARA TENSORFLOW ===
X_train (60%): (42000, 28, 28, 1) | y_train: (42000,)
X_val   (20%): (14000, 28, 28, 1)   | y_val:   (14000,)
X_test  (20%): (14000, 28, 28, 1)  | y_test:  (14000,)
Rango de píxeles: [0.0, 1.0] | Tipo de dato: float32


Elegimos  TensorFlow/Keras porque nos permite implementar los modelos requeridos de manera sencilla y clara,  facilitando el desarrollo del MLP y la CNN, tambien el entrenamiento de los modelos, los resultados  de las métricas y el análisis de las curvas de aprendizaje.

### 1.2 Cargar del dataset.

ELegimos el dataset KMNIST (Kuzushiji-MNIST) es un dataset de visión por computadora compuesto por imágenes en escala de grises de 28x28 píxeles con caracteres japoneses tradicionales (Kuzushiji)


In [2]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Cargar los paquetes desde el disco local
X_crudo_train = np.load('kmnist-train-imgs.npz')['arr_0']
y_crudo_train = np.load('kmnist-train-labels.npz')['arr_0']
X_crudo_test  = np.load('kmnist-test-imgs.npz')['arr_0']
y_crudo_test  = np.load('kmnist-test-labels.npz')['arr_0']

# 2. Juntar todo el lote (70.000 piezas)
X_total = np.concatenate([X_crudo_train, X_crudo_test], axis=0)
y_total = np.concatenate([y_crudo_train, y_crudo_test], axis=0)

# 3. División estratificada en 60% / 20% / 20%
# Primer corte: 60% Train y 40% Temporal
X_train, X_temp, y_train, y_temp = train_test_split(
    X_total, y_total, test_size=0.40, random_state=42, stratify=y_total
)

# Segundo corte: 20% Validación y 20% Prueba
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# 4. Acondicionamiento para TensorFlow:
# - Normalización a rango 0.0 - 1.0 (float32)
# - Agregado del canal de profundidad (28, 28, 1)
X_train = (X_train / 255.0).astype(np.float32)[..., np.newaxis]
X_val   = (X_val   / 255.0).astype(np.float32)[..., np.newaxis]
X_test  = (X_test  / 255.0).astype(np.float32)[..., np.newaxis]

# 5. Verificación de control de calidad
print("=== MATERIA PRIMA LISTA PARA LA RED NEURONAL ===")
print(f"Entrenamiento (60%): {X_train.shape} | Etiquetas: {y_train.shape}")
print(f"Validación    (20%): {X_val.shape}   | Etiquetas: {y_val.shape}")
print(f"Prueba        (20%): {X_test.shape}  | Etiquetas: {y_test.shape}")
print(f"Rango de valores: [{X_train.min()}, {X_train.max()}] | Tipo: {X_train.dtype}")

FileNotFoundError: [Errno 2] No such file or directory: 'kmnist-train-imgs.npz'



### 1.3 Análisis exploratorio

*[Completar durante el desarrollo]*

### 1.4 Preprocesamiento

*[Completar durante el desarrollo]*

### 1.5 Partición Train / Dev / Test

*[Completar durante el desarrollo]*

### 1.6 Modelo baseline

*[Completar durante el desarrollo]*

### 1.7 Métricas y evaluación inicial

*[Completar durante el desarrollo]*

### 1.8 Conclusiones de la Etapa 1

*[Completar durante el desarrollo]*

# ETAPA 2 — Análisis de errores y red neuronal multicapa

### Contenido principal
- Análisis de errores.
- Red neuronal multicapa (MLP).
- Regularización.
- Diagnóstico y mitigación del sobreajuste.

### Arquitectura
**MLP (red densa)**

---

### 2.1 Análisis de errores del baseline

*[Completar durante el desarrollo]*

### 2.2 Diagnóstico: sesgo y varianza

*[Completar durante el desarrollo]*

### 2.3 Construcción de la MLP

*[Completar durante el desarrollo]*

### 2.4 Entrenamiento

*[Completar durante el desarrollo]*

### 2.5 Regularización y mitigación del sobreajuste

*[Completar durante el desarrollo]*

### 2.6 Evaluación y comparación con baseline

*[Completar durante el desarrollo]*

### 2.7 Conclusiones de la Etapa 2

*[Completar durante el desarrollo]*

# ETAPA 3 — CNN / Red densa avanzada

### Contenido principal
- Selección de arquitectura según el tipo de dataset.
- Para datasets de imágenes: CNN.
- Alternativamente: red densa avanzada.
- Pipeline completo.

### Arquitectura
**CNN o MLP avanzado**

---

### 3.1 Justificación de la arquitectura

*[Completar durante el desarrollo]*

### 3.2 Preparación de los datos para la arquitectura elegida

*[Completar durante el desarrollo]*

### 3.3 Construcción del modelo

*[Completar durante el desarrollo]*

### 3.4 Entrenamiento

*[Completar durante el desarrollo]*

### 3.5 Evaluación

*[Completar durante el desarrollo]*

### 3.6 Comparación con modelos anteriores

*[Completar durante el desarrollo]*

### 3.7 Conclusiones de la Etapa 3

*[Completar durante el desarrollo]*

# ETAPA 4 — Modelo generativo

### Contenido principal
- Construcción de un modelo generativo.
- VAE o GAN/DCGAN según corresponda al dataset.
- Análisis del espacio latente.

### Arquitectura
**VAE / DCGAN**

---

### 4.1 Justificación del modelo generativo elegido

*[Completar durante el desarrollo]*

### 4.2 Preparación de los datos

*[Completar durante el desarrollo]*

### 4.3 Construcción del modelo

*[Completar durante el desarrollo]*

### 4.4 Entrenamiento

*[Completar durante el desarrollo]*

### 4.5 Exploración del espacio latente

*[Completar durante el desarrollo]*

### 4.6 Resultados

*[Completar durante el desarrollo]*

### 4.7 Conclusiones de la Etapa 4

*[Completar durante el desarrollo]*

# ETAPA 5 — Cierre y comparación de modelos

### Contenido principal
- Comparación de los modelos desarrollados.
- Análisis de resultados.
- Análisis ético de sesgos.
- Conclusiones finales.

### Arquitectura
**Resumen integral**

---

### 5.1 Comparación de modelos

*[Completar durante el desarrollo]*

### 5.2 Comparación de métricas

*[Completar durante el desarrollo]*

### 5.3 Análisis de errores y resultados

*[Completar durante el desarrollo]*

### 5.4 Análisis ético y de sesgos

*[Completar durante el desarrollo]*

### 5.5 Conclusiones finales

*[Completar durante el desarrollo]*

---
# Referencias

*[Agregar las fuentes utilizadas durante el desarrollo del TP]*